# 02 Train DeepVoid on your own simulation

A template. Replace the loading cell with your own data and run the rest. `01_quickstart.ipynb` runs the same steps on a small sample if you want to see them work first.

**What you need**

- The **matter density** on a grid covering the whole periodic box, for the labels. Cells must be smaller than 1 h⁻¹ Mpc for the default label convention (for example 512³ cells in a 205 h⁻¹ Mpc box).
- The **tracers** the model should work from (halos, subhalos or mock galaxies): positions in h⁻¹ Mpc, and masses if you want to select the most massive ones or weight by mass. A tracer density already on the grid works too.

**What you get**

A model that labels each cell void, wall, filament or halo from the tracer field alone, with scores on a slab of the box held out from training.

Use a GPU runtime. At 512³ the label step needs a High-RAM runtime (about 6 GB at its peak), and the grids are held in GPU memory during training.

In [ ]:
# Get the code. While the repository is private this needs a Colab secret
# named GITHUB_TOKEN; once it is public the try-block does nothing.
url = "https://github.com/skumagai3/deepvoid.git"
try:
    from google.colab import userdata
    url = url.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
!rm -rf /content/deepvoid && git clone -q {url} /content/deepvoid
!pip -q install /content/deepvoid matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Optional: keep the project on Google Drive so it survives the session.
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/deepvoid_projects/my_simulation"

## 1. Load your data

Replace this cell. It should end with:

- `matter_density`: array (n, n, n), mass or particle counts per cell, any units
- `positions`: array (N, 3) in h⁻¹ Mpc, inside [0, box_size)
- `masses`: array (N,), or `None`
- `box_size`: in h⁻¹ Mpc

Axis order is (x, y, z) for both the grid and the positions. If your particles are not gridded yet, `deepvoid.grid.cic` does cloud-in-cell assignment.

In [ ]:
import numpy as np

box_size = 205.0
matter_density = np.load("/path/to/matter_density_512.npy")
halos = np.load("/path/to/halo_catalog.npz")
positions, masses = halos["pos"], halos["mass"]

## 2. Build the project

Each tracer sample is described by its mean separation λ = n^(−1/3) in h⁻¹ Mpc, where n is its number density. `spacing=5.0` keeps the most massive tracers so that λ = 5; without `spacing`, all tracers are used. Sparser samples are harder: more of the structure has no tracer near it.

Adding several spacings lets one model handle all of them; the model in the repository README was trained this way, on spacings 3, 5, 7 and 10.

In [ ]:
from deepvoid import Project

project = Project(PROJECT)
n = matter_density.shape[0]

project.add_density("dm", matter_density, box_size=box_size, tracer="dm")
project.add_labels("labels", density="dm")            # T-web: sigma_eff = 1 h^-1 Mpc, threshold 0.65
for spacing in (3.0, 5.0, 7.0, 10.0):
    project.add_tracers(f"halos/L{spacing:g}", positions, box_size=box_size, n=n,
                        masses=masses, spacing=spacing)
print(project.summary())

Check the summary before training. For reference, the TNG300-3 labels at the default convention are 65% void, 26% wall, 9% filament and 0.5% halo. Very different fractions usually mean the density is not what you think it is (wrong units of length, a non-periodic sub-box, or a field that is not the total matter).

In [ ]:
from deepvoid import figures

fig = figures.fields_figure({"matter": project.load("dm").data,
                             "halos, spacing 10": project.load("halos/L10").data},
                            project.load("labels").data, index=n // 2, box_size=box_size)

## 3. Settings

These are the settings of our best model so far. `configs/template.yaml` explains each one. The defaults assume a grid of about 512³; for a smaller grid, reduce `training.crop` and `training.val_crop` so they fit inside the slabs (the error message says by how much).

In [ ]:
cfg = project.config(
    "first_run",
    inputs=["halos/L3", "halos/L5", "halos/L7", "halos/L10"],
    labels="labels",
    data={"smoothing": {"sigma_spacing": 0.5}},        # second channel: the input smoothed over half its spacing
    model={"depth": 5},
    training={"epochs": 60, "val_batches": 16,
              "optimizer": {"weight_decay": 1.0e-3},
              "loss": {"class_weights": "auto"},
              "early_stopping": {"monitor": "val_mcc"}},
)

## 4. Train

Rerun this cell after a disconnect and training continues from the last finished epoch. Progress is in `log.csv` in the run folder, one row per epoch.

In [ ]:
from deepvoid.train import run

out = run(cfg)

## 5. Results

The test scores come from a prediction of the whole box, scored only on the test slab.

In [ ]:
from deepvoid.runs import Run

r = Run(out)
for name, s in r.scores().items():
    print(f"{name}: four-class MCC {s['mcc']:.3f}, void MCC {s['void_mcc']:.3f}, void F1 {s['void_f1']:.3f}")
fig = figures.training_curves([r], keys=("val_mcc", "val_void_mcc", "val_loss"))

In [ ]:
name = "halos/L10"
scores = r.predict(name)                  # class scores for every cell, shape (n, n, n, 4)
test = r.regions["test"]
fig = figures.prediction_figure(r.input(name), r.labels(), scores, index=(test.start + test.stop) // 2,
                                box_size=box_size, input_label=name)

## 6. Save a labelled box

The predicted class of a cell is the one with the highest score; `scores[..., 0]` is the void score.

In [ ]:
from deepvoid import Volume, save_volume

predicted = Volume(scores.argmax(-1).astype(np.uint8), box_size,
                   {"field": "mask", "labels": "deepvoid", "run": r.name, "input": name})
save_volume(predicted, f"{PROJECT}/predicted_{name.replace('/', '_')}.npz")